In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os, glob
from matplotlib.ticker import MultipleLocator

In [ ]:
import matplotlib as mpl
from matplotlib import font_manager
# Custom font path.
font_path = '../../data/Arial.ttf'

# Add the font to matplotlib.
font_manager.fontManager.addfont(font_path)

# Get the registered font name.
custom_font = font_manager.FontProperties(fname=font_path)
font_name = custom_font.get_name()
# Set the global font.
mpl.rcParams['font.family'] = font_name

In [ ]:
city = 'LosAngeles'
files = glob.glob(f'../../data/regression_outputs_new/Fold/US/{city}/SV/*/results.csv')
files = sorted(files)
len(files)

In [ ]:
targets = [
    'logcrime', 'logpetty', 'walkbike_per_cbg', 'publictrans_per_cbg', 'drove_alone_per_cbg', 'estvmiles' ,
    'estpmiles', 'estvtrp', 'estptrp', 'obesitycru', 'diabetescr', 'lpacrudepr', 
    'mhlthcrude','phlthcrude','cancercrud','logincome','povertyline_below100','povertyline_below200'
]

ls = []
for i, file in enumerate(files):
    model = file.split('/')[-2]
    tmp = pd.read_csv(file)
    tmp.set_index('target', inplace=True)
    tmp = tmp.loc[targets]
    if 'avg_r2' in tmp.columns:
        tmp = tmp[['avg_r2']]
        tmp = tmp.rename(columns={'avg_r2': model})
    else:
        tmp = tmp[['r2']]
        tmp = tmp.rename(columns={'r2': model})
    ls.append(tmp)
df = pd.concat(ls, axis=1)
df.reset_index(inplace=True)
df

In [ ]:
columns = [i.split('-')[-1] if 'Multi' in i else i for i in df.columns.tolist()]
df.columns = columns
df

In [ ]:
sv_df = df.melt(value_vars=columns[1:], value_name='R2', var_name='Epoch')
sv_df

In [ ]:
files = glob.glob(f'../../data/regression_outputs_new/Fold/US/LosAngeles/RS/*/results.csv')
files = sorted(files)
len(files)

In [ ]:
targets = [
    'logcrime', 'logpetty', 'walkbike_per_cbg', 'publictrans_per_cbg', 'drove_alone_per_cbg', 'estvmiles' ,
    'estpmiles', 'estvtrp', 'estptrp', 'obesitycru', 'diabetescr', 'lpacrudepr', 
    'mhlthcrude','phlthcrude','cancercrud','logincome','povertyline_below100','povertyline_below200'
]

ls = []
for i, file in enumerate(files):
    model = file.split('/')[-2]
    tmp = pd.read_csv(file)
    tmp.set_index('target', inplace=True)
    tmp = tmp.loc[targets]
    if 'avg_r2' in tmp.columns:
        tmp = tmp[['avg_r2']]
        tmp = tmp.rename(columns={'avg_r2': model})
    else:
        tmp = tmp[['r2']]
        tmp = tmp.rename(columns={'r2': model})
    ls.append(tmp)
df = pd.concat(ls, axis=1)
df.reset_index(inplace=True)
df

In [ ]:
df.columns = [
    'target', 'ep149', 'ep019', 'ep199', 'ep249', 'ep029', 'ep299', 'ep039', 'ep049','ep009', 'ep099',
]

df = df[sorted(df.columns)]
df

In [ ]:
rs_df = df.melt(value_vars=[
    'ep009', 
    'ep019', 
    'ep029', 
    'ep039', 
    'ep049', 
    'ep099',
    'ep149', 
    'ep199', 
    'ep249', 
    'ep299', 
    ], value_name='R2', var_name='Epoch')
rs_df['Epoch'] = rs_df['Epoch'].str.replace('ep', '').astype(int)
rs_df

In [ ]:
sv_df['Epoch'] = sv_df['Epoch'].str.replace('ep', '').astype(int)
sv_df

In [ ]:
colors = ['#8178b2', '#f5a050']

fig, ax = plt.subplots(1,1, figsize=(12, 8))

# RS
sns.lineplot(data=rs_df, x='Epoch', y='R2', ax=ax, linewidth=2, label='Satellite-based models', color=colors[0], marker=None)
mean_rs = rs_df.groupby('Epoch')['R2'].mean().reset_index()
ax.scatter(mean_rs['Epoch'], mean_rs['R2'], color=colors[0], s=100, alpha=0.7, zorder=3, label='_nolegend_', edgecolor='white')

# SV
sns.lineplot(data=sv_df, x='Epoch', y='R2', ax=ax, linewidth=2, label='Street view-based models', color=colors[1], marker=None)
mean_sv = sv_df.groupby('Epoch')['R2'].mean().reset_index()
ax.scatter(mean_sv['Epoch'], mean_sv['R2'], color=colors[1], s=100, alpha=0.7, zorder=3, label='_nolegend_', edgecolor='white')

ax.set_xlabel('Pre-training epoch', fontsize=18)
ax.set_ylabel('$R^2$', fontsize=18)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)  # keep left spine
ax.spines['bottom'].set_visible(True)  # keep bottom spine
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

# Set y-tick spacing to 20.
ax.yaxis.set_major_locator(MultipleLocator(0.05))

ax.tick_params(axis='x', labelsize=16)  # x-tick label size
ax.tick_params(axis='y', labelsize=16)  # y-tick label size

plt.legend(
    loc='upper left', 
    fontsize=18, 
    frameon=False, 
    markerscale=2,
    handlelength=1.5,
    handletextpad=0.5,
    borderpad=0.5,
    labelspacing=0.5,
    title_fontsize=16
)
plt.savefig(
    '../../data/figure_assets/pretraining_epoch.pdf',
    bbox_inches='tight',
    dpi=300
)
plt.show()